# Testing the DLOmix Local Intensity Model Integration

This notebook exercises the DLOmix integration added to Oktoberfest: a `Rescoring` job where fragment
intensity predictions come from a **local, pretrained DLOmix `InferencePipeline`** (via the
`dlomix_intensity` model key) instead of a Koina model. iRT predictions are still fetched from Koina as
usual.

It walks through:
1. (Optional) downloading the standard Oktoberfest example dataset from Zenodo, for anyone who doesn't
   already have their own MaxQuant search results + spectra to test with.
2. Building and running a `Rescoring` config that points `dlomix_intensity` at a local DLOmix model.

## 1- Import necessary python packages

In [1]:
import json
import os
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display
from tqdm import tqdm

from oktoberfest.runner import run_job

Using TensorFlow Backend for DLOmix. To change the backend, set the DLOMIX_BACKEND environment variable to tensorflow or pytorch and re-import DLOmix.


2026-08-17 18:37:33.636060: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-17 18:37:33.636254: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-17 18:37:33.928899: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-17 18:37:34.524088: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-17 18:37:47.756703: W tensorflow/compiler/tf2

## 2- (Optional) Download example files from Zenodo

This is the same general-purpose Oktoberfest example dataset used in the main tutorial
(`Oktoberfest Tutorial.ipynb`): MaxQuant search results plus raw spectra files, ~2.7GB in total.

**Skip this section** if you already have search results + spectra to test DLOmix with (e.g. the tomato
dataset referenced in section 3 below) — it's only here so this notebook is runnable standalone.

### A- Get the current directory and set the file name

In [2]:
download_dir = os.getcwd()
download_file = os.path.join(download_dir, 'Oktoberfest_input.zip')
url = 'https://zenodo.org/record/7613029/files/Oktoberfest_input.zip'

download = False  # set this to True to (re)download; leave False if you already have the file/data

### B- Download and extract files from Zenodo to the same directory

In [3]:
if download:
    with tqdm(unit="B", total=2739196307, unit_scale=True, unit_divisor=1000, miniters=1, desc=url.split("/")[-1]) as t:
        urllib.request.urlretrieve(
            url=url,
            filename=download_file,
            reporthook=lambda blocks, block_size, _: t.update(blocks * block_size - t.n),
        )
    shutil.unpack_archive(download_file, download_dir)

### C- Check downloaded files

In [4]:
input_dir = download_file[:-4]

if os.path.isdir(input_dir):
    print(f'Downloaded data is stored in {input_dir}\nContents:')
    print(os.listdir(input_dir))
else:
    print(
        f'{input_dir} not found. Set download = True above and re-run this section, '
        'or just point directly at your own search results/spectra in section 3 below.'
    )

/cmnfs/home/v.giurcoiu/oktoberfest_new/oktoberfest/tutorials/Oktoberfest_input not found. Set download = True above and re-run this section, or just point directly at your own search results/spectra in section 3 below.


## 3- Configure and run the DLOmix rescoring job

A few things that are specific to the DLOmix integration compared to a normal Koina-based rescoring job:

- `models.dlomix_intensity` points at a local, pretrained DLOmix `InferencePipeline` directory (as opposed
  to a Koina model name string like `Prosit_2020_intensity_HCD`, which would go under `models.intensity`).
- `models.irt` is still resolved via Koina as usual — DLOmix here only replaces the intensity predictor.
- Whenever `dlomix_intensity` is set, Oktoberfest automatically pins `numThreads` to **1** internally,
  regardless of what's configured below, because DLOmix's TensorFlow pipeline is not safe to fork into
  multiple worker processes (see `Config.num_threads` in `oktoberfest/utils/config.py`). `numThreads` is
  kept in the config below for documentation only.

By default the paths below point at the tomato dataset already used to validate this integration locally.
Swap `search_results` / `spectra` for `input_dir` from the Zenodo download above (with
`spectra_type: "raw"`) if you used section 2 instead.

In [ ]:
search_results = "/cmnfs/home/v.giurcoiu/test_data/tomato_dataset_example/msms.txt"
spectra = "/cmnfs/home/v.giurcoiu/test_data/tomato_dataset_example/5407_GC4_063119_S00_U4_R1.mzML"
# TODO: replace this with the huggingface model
dlomix_model_path = "/cmnfs/data/proteomics/dlomix_data/artifacts/intensity_model"
output_dir = "./output_dlomix_test"

task_config_dlomix_rescoring = {
    "type": "Rescoring",
    "numThreads": 1, 
    "output": output_dir,
    "inputs": {
        "search_results": search_results,
        "search_results_type": "maxquant",
        "spectra": spectra,
        "spectra_type": "mzml",
    },
    "models": {
        "irt": "Prosit_2019_irt",
        "dlomix_intensity": dlomix_model_path,
    },
}
task_config_dlomix_rescoring

{'type': 'Rescoring',
 'numThreads': 1,
 'output': './output_dlomix_test',
 'inputs': {'search_results': '/cmnfs/home/v.giurcoiu/test_data/tomato_dataset_example/msms.txt',
  'search_results_type': 'maxquant',
  'spectra': '/cmnfs/home/v.giurcoiu/test_data/tomato_dataset_example/5407_GC4_063119_S00_U4_R1.mzML',
  'spectra_type': 'mzml'},
 'models': {'irt': 'Prosit_2019_irt',
  'dlomix_intensity': '/cmnfs/data/proteomics/dlomix_data/artifacts/intensity_model'}}

#### Save config as json

In [6]:
with open('./dlomix_rescoring_config.json', 'w') as fp:
    json.dump(task_config_dlomix_rescoring, fp, indent=2)

#### Run the rescoring job

Watch the log for `Using DLOmix intensity model` — this confirms DLOmix (not Koina) produced the
intensity predictions. Since DLOmix runs single-threaded, predicting intensities for tens of thousands of
PSMs can take a while (roughly minutes, not seconds).

In [7]:
run_job("./dlomix_rescoring_config.json")

2026-08-17 18:39:58,088 - INFO - oktoberfest.utils.config::read Reading configuration from ./dlomix_rescoring_config.json
2026-08-17 18:39:58,095 - INFO - oktoberfest.runner::run_job Oktoberfest version 0.11.0
Copyright 2026, Wilhelmlab at Technical University of Munich
2026-08-17 18:39:58,096 - INFO - oktoberfest.runner::run_job Job executed with the following config:
2026-08-17 18:39:58,145 - INFO - oktoberfest.runner::run_job {
    "type": "Rescoring",
    "numThreads": 1,
    "output": "./output_dlomix_test",
    "inputs": {
        "search_results": "/cmnfs/home/v.giurcoiu/test_data/tomato_dataset_example/msms.txt",
        "search_results_type": "maxquant",
        "spectra": "/cmnfs/home/v.giurcoiu/test_data/tomato_dataset_example/5407_GC4_063119_S00_U4_R1.mzML",
        "spectra_type": "mzml"
    },
    "models": {
        "irt": "Prosit_2019_irt",
        "dlomix_intensity": "/cmnfs/data/proteomics/dlomix_data/artifacts/intensity_model"
    }
}
2026-08-17 18:39:58,147 - INFO -

/cmnfs/home/v.giurcoiu/miniconda3/envs/okt_py310/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


2026-08-17 18:43:39,455 - INFO - oktoberfest.preprocessing.preprocessing::annotate_spectral_library Finished annotating.
2026-08-17 18:43:42,358 - INFO - oktoberfest.utils.config::_load_dlomix_pipeline Loading DLOmix pipeline from /cmnfs/data/proteomics/dlomix_data/artifacts/intensity_model (pid 2890579)


/cmnfs/home/v.giurcoiu/miniconda3/envs/okt_py310/lib/python3.10/site-packages/dlomix/data/processing/processors.py:377: UserWarning: The unknown token 'X' is already present in the provided alphabet with index 1. If you prefer the default behavior, consider removing it from the alphabet and it will have the default index of 1.
  warnings.warn(
2026-08-17 18:43:42.554569: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-08-17 18:43:42.554644: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:129] retrieving CUDA diagnostic information for host: kraken.exbio.wzw.tum.de
2026-08-17 18:43:42.554660: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:136] hostname: kraken.exbio.wzw.tum.de
2026-08-17 18:43:42.554836: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:159] libcuda reported version is: 570.207.0
2026-08-17 18:43:42.554907: I externa

2026-08-17 18:43:45,412 - INFO - oktoberfest.predict.predictor::from_config Using DLOmix intensity model


...

Casting the dataset:   0%|          | 0/74703 [00:00<?, ? examples/s]

292/292 [==============================] - 60s 206ms/step
2026-08-17 18:45:37,129 - INFO - oktoberfest.predict.dlomix::predict Successfully predicted intensities
2026-08-17 18:45:56,934 - INFO - oktoberfest.predict.predictor::from_config Using model Prosit_2019_irt via Koina


Prosit_2019_irt::   0%|          | 0/75 [00:00<?, ?it/s]

/cmnfs/home/v.giurcoiu/miniconda3/envs/okt_py310/lib/python3.10/site-packages/spectrum_fundamentals/metrics/similarity.py:243: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  scipy.stats.pearsonr(obs, pred)[0] if method == "pearson" else scipy.stats.spearmanr(obs, pred)[0]
/cmnfs/home/v.giurcoiu/miniconda3/envs/okt_py310/lib/python3.10/site-packages/spectrum_fundamentals/metrics/percolator.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  retention_time_df = retention_time_df.groupby("rt_bin_index", group_keys=False).apply(


2026-08-17 18:46:42,356 - INFO - oktoberfest.runner::run_rescoring Merging input tab files for rescoring without peptide property prediction
2026-08-17 18:46:43,715 - INFO - oktoberfest.runner::run_rescoring Merging input tab files for rescoring with peptide property prediction
2026-08-17 18:46:49,159 - INFO - oktoberfest.rescore.rescore::rescore_with_percolator Starting percolator with command percolator --weights output_dlomix_test/results/percolator/original.percolator.weights.csv                         --num-threads 3                         --subset-max-train 500000                         --post-processing-tdc                         --testFDR 0.01                         --trainFDR 0.01                         --results-psms output_dlomix_test/results/percolator/original.percolator.psms.txt                         --decoy-results-psms output_dlomix_test/results/percolator/original.percolator.decoy.psms.txt                         --results-peptides output_dlomix_test/results/pe